# Phase 2: Content-Based Movie Similarity

**Goal**: Given a movie title, return N similar movies using two approaches:
- **Approach A**: TF-IDF + cosine similarity
- **Approach B**: Dense embeddings + FAISS vector search

**Deliverables**:
- Both similarity approaches implemented and compared
- Side-by-side results in notebook
- Results saved to CSV for analysis
- Embeddings and FAISS index saved for Phase 3

## 1. Setup & Load Phase 1 Checkpoints

In [1]:
import os
import pandas as pd
import numpy as np
import duckdb
import sqlite3
import json
from pathlib import Path
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Auto-install required packages
try:
    import sentence_transformers
except ImportError:
    import sys
    import subprocess
    print("Installing sentence-transformers...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sentence-transformers'])
    import sentence_transformers

try:
    import faiss
except ImportError:
    import sys
    import subprocess
    print("Installing faiss-cpu...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faiss-cpu'])
    import faiss

from sentence_transformers import SentenceTransformer

# Setup directories
DATA_DIR = Path('data')
CHECKPOINTS_DIR = Path('checkpoints')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoints directory: {CHECKPOINTS_DIR}")
print(f"Results directory: {RESULTS_DIR}")

Working directory: c:\Users\Meenhaz\Claude Workspace\Movie Search Recommendations
Data directory: data
Checkpoints directory: checkpoints
Results directory: results


## 2. Load Phase 1 Data

In [2]:
def load_phase1_from_duckdb(db_path):
    """Load persisted Phase 1 tables from DuckDB."""
    conn = duckdb.connect(str(db_path), read_only=True)
    required_tables = {'ml_movies', 'ml_ratings', 'ml_tags', 'imdb_basics', 'imdb_ratings'}
    existing_tables = {row[0] for row in conn.execute('SHOW TABLES').fetchall()}
    missing_tables = sorted(required_tables - existing_tables)
    if missing_tables:
        conn.close()
        raise RuntimeError(f"DuckDB checkpoint is missing tables: {', '.join(missing_tables)}")
    
    return conn, {
        'ml_movies': conn.query('SELECT * FROM ml_movies').to_df(),
        'ml_ratings': conn.query('SELECT movieId, rating FROM ml_ratings').to_df(),
        'ml_tags': conn.query('SELECT movieId, tag FROM ml_tags').to_df(),
        'imdb_basics': conn.query('SELECT * FROM imdb_basics').to_df(),
        'imdb_ratings': conn.query('SELECT * FROM imdb_ratings').to_df(),
    }


def load_phase1_from_files(data_dir):
    """Fallback loader for when the DuckDB checkpoint is locked or incomplete."""
    movielens_path = data_dir / 'movielens'
    imdb_path = data_dir / 'imdb'
    required_files = [
        movielens_path / 'movies.csv',
        movielens_path / 'ratings.csv',
        movielens_path / 'tags.csv',
        imdb_path / 'title.basics.tsv.gz',
        imdb_path / 'title.ratings.tsv.gz',
    ]
    missing_files = [str(path) for path in required_files if not path.exists()]
    if missing_files:
        raise FileNotFoundError('Missing raw data files: ' + ', '.join(missing_files))
    
    return {
        'ml_movies': pd.read_csv(movielens_path / 'movies.csv'),
        'ml_ratings': pd.read_csv(movielens_path / 'ratings.csv', usecols=['movieId', 'rating']),
        'ml_tags': pd.read_csv(movielens_path / 'tags.csv', usecols=['movieId', 'tag']),
        'imdb_basics': pd.read_csv(imdb_path / 'title.basics.tsv.gz', sep='\t', compression='gzip', dtype={'tconst': str}),
        'imdb_ratings': pd.read_csv(imdb_path / 'title.ratings.tsv.gz', sep='\t', compression='gzip', dtype={'tconst': str}),
    }


# Load Phase 1 data. Prefer the checkpoint, but fall back to raw files if Windows has locked DuckDB.
db_path = CHECKPOINTS_DIR / 'movies.duckdb'
try:
    try:
        duck_conn.close()
    except NameError:
        pass
    except Exception:
        pass
    
    duck_conn, phase1_data = load_phase1_from_duckdb(db_path)
    print(f"✓ DuckDB connected: {db_path}")
except Exception as exc:
    print(f"⚠ DuckDB checkpoint unavailable: {exc}")
    print("Loading Phase 1 data directly from raw files instead...")
    duck_conn = None
    phase1_data = load_phase1_from_files(DATA_DIR)

ml_movies = phase1_data['ml_movies']
ml_ratings = phase1_data['ml_ratings']
ml_tags = phase1_data['ml_tags']
imdb_basics = phase1_data['imdb_basics']
imdb_ratings = phase1_data['imdb_ratings']

print(f"✓ Loaded {len(ml_movies)} MovieLens movies")
print(f"✓ Loaded {len(ml_ratings)} MovieLens ratings")
print(f"✓ Loaded {len(ml_tags)} MovieLens tags")
print(f"✓ Loaded {len(imdb_basics)} IMDB titles")

⚠ DuckDB checkpoint unavailable: DuckDB checkpoint is missing tables: imdb_basics, imdb_ratings, ml_movies, ml_ratings, ml_tags
Loading Phase 1 data directly from raw files instead...
✓ Loaded 62423 MovieLens movies
✓ Loaded 25000095 MovieLens ratings
✓ Loaded 1093360 MovieLens tags
✓ Loaded 12513043 IMDB titles


## 3. Feature Engineering

### 3.1 Extract and Combine Features

In [3]:
def engineer_features(movies_df, ratings_df, tags_df):
    """
    Engineer combined text features for similarity.
    Combines: genres, tags, and popularity signal.
    """
    # Calculate mean rating per movie
    rating_stats = ratings_df.groupby('movieId').agg({
        'rating': ['mean', 'count']
    }).reset_index()
    rating_stats.columns = ['movieId', 'rating_mean', 'rating_count']
    
    # Combine tags into single text per movie
    tags_text = tags_df.groupby('movieId')['tag'].apply(
        lambda x: ' '.join(x.astype(str))
    ).reset_index()
    tags_text.columns = ['movieId', 'tags_combined']
    
    # Merge all features
    movies_enriched = movies_df.copy()
    movies_enriched = movies_enriched.merge(rating_stats, on='movieId', how='left')
    movies_enriched = movies_enriched.merge(tags_text, on='movieId', how='left')
    
    # Fill missing values
    movies_enriched['tags_combined'] = movies_enriched['tags_combined'].fillna('')
    movies_enriched['rating_mean'] = movies_enriched['rating_mean'].fillna(0)
    movies_enriched['rating_count'] = movies_enriched['rating_count'].fillna(0)
    
    # Create combined text field for similarity
    movies_enriched['combined_text'] = (
        movies_enriched['genres'].fillna('') + ' ' +
        movies_enriched['tags_combined'].fillna('')
    ).str.strip()
    
    return movies_enriched

print("Engineering features...")
movies_enriched = engineer_features(ml_movies, ml_ratings, ml_tags)
print(f"✓ Created enriched feature set: {movies_enriched.shape}")
print(f"\nSample movie features:")
print(movies_enriched[['movieId', 'title', 'genres', 'rating_mean', 'combined_text']].head(3))

Engineering features...
✓ Created enriched feature set: (62423, 7)

Sample movie features:
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  rating_mean  \
0  Adventure|Animation|Children|Comedy|Fantasy     3.893708   
1                   Adventure|Children|Fantasy     3.251527   
2                               Comedy|Romance     3.142028   

                                       combined_text  
0  Adventure|Animation|Children|Comedy|Fantasy Ow...  
1  Adventure|Children|Fantasy Robin Williams time...  
2  Comedy|Romance funny best friend duringcredits...  


## 4. Approach A: TF-IDF + Cosine Similarity

In [4]:
class TFIDFSimilarity:
    """TF-IDF based movie similarity."""
    
    def __init__(self, movies_df):
        self.movies_df = movies_df.copy()
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            lowercase=True,
            stop_words='english',
            ngram_range=(1, 2)
        )
        
        # Fit vectorizer on combined text
        print("Fitting TF-IDF vectorizer...")
        self.tfidf_matrix = self.vectorizer.fit_transform(self.movies_df['combined_text'])
        print(f"✓ TF-IDF matrix shape: {self.tfidf_matrix.shape}")
    
    def find_similar(self, movie_title, n=10):
        """Find N similar movies by title."""
        # Find movie
        movie_matches = self.movies_df[self.movies_df['title'].str.contains(movie_title, case=False, na=False)]
        
        if len(movie_matches) == 0:
            return None, f"Movie '{movie_title}' not found"
        
        # Use first match
        seed_movie = movie_matches.iloc[0]
        seed_idx = self.movies_df[self.movies_df['movieId'] == seed_movie['movieId']].index[0]
        
        # Calculate similarities
        similarities = cosine_similarity(self.tfidf_matrix[seed_idx], self.tfidf_matrix)[0]
        
        # Get top N (excluding the seed movie itself)
        top_indices = np.argsort(similarities)[::-1][1:n+1]
        
        results = []
        for idx in top_indices:
            movie = self.movies_df.iloc[idx]
            results.append({
                'rank': len(results) + 1,
                'movieId': movie['movieId'],
                'title': movie['title'],
                'genres': movie['genres'],
                'similarity_score': similarities[idx],
                'rating_mean': movie['rating_mean']
            })
        
        return pd.DataFrame(results), seed_movie

# Initialize TF-IDF similarity
tfidf_sim = TFIDFSimilarity(movies_enriched)

Fitting TF-IDF vectorizer...
✓ TF-IDF matrix shape: (62423, 5000)


## 5. Approach B: Dense Embeddings + FAISS

In [5]:
class EmbeddingSimilarity:
    """Dense embedding based movie similarity using FAISS."""
    
    def __init__(self, movies_df, model_name='all-MiniLM-L6-v2'):
        self.movies_df = movies_df.copy()
        self.model_name = model_name
        
        # Load embedding model
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        print(f"✓ Model loaded. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        
        # Compute embeddings
        print(f"Computing embeddings for {len(self.movies_df)} movies...")
        embeddings = self.model.encode(
            self.movies_df['combined_text'].tolist(),
            show_progress_bar=True,
            convert_to_numpy=True
        )
        
        # L2 normalize
        embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
        self.embeddings = embeddings.astype('float32')
        
        # Build FAISS index
        print("Building FAISS index...")
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(self.embeddings)
        print(f"✓ FAISS index built with {self.index.ntotal} vectors")
    
    def find_similar(self, movie_title, n=10):
        """Find N similar movies by title."""
        # Find movie
        movie_matches = self.movies_df[self.movies_df['title'].str.contains(movie_title, case=False, na=False)]
        
        if len(movie_matches) == 0:
            return None, f"Movie '{movie_title}' not found"
        
        # Use first match
        seed_movie = movie_matches.iloc[0]
        seed_idx = self.movies_df[self.movies_df['movieId'] == seed_movie['movieId']].index[0]
        
        # Query FAISS
        query_embedding = self.embeddings[seed_idx:seed_idx+1]
        distances, indices = self.index.search(query_embedding, n + 1)
        
        results = []
        for rank, idx in enumerate(indices[0][1:n+1]):
            movie = self.movies_df.iloc[idx]
            results.append({
                'rank': rank + 1,
                'movieId': movie['movieId'],
                'title': movie['title'],
                'genres': movie['genres'],
                'similarity_score': distances[0][rank + 1],
                'rating_mean': movie['rating_mean']
            })
        
        return pd.DataFrame(results), seed_movie
    
    def save_embeddings(self, path):
        """Save embeddings for Phase 3."""
        np.save(path / 'embeddings.npy', self.embeddings)
        self.movies_df[['movieId', 'title']].to_csv(path / 'movie_ids.csv', index=False)
        faiss.write_index(self.index, str(path / 'faiss_index.bin'))
        print(f"✓ Saved embeddings to {path}")

# Initialize embedding similarity
embedding_sim = EmbeddingSimilarity(movies_enriched)

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2472.05it/s]


✓ Model loaded. Embedding dimension: 384
Computing embeddings for 62423 movies...


Batches: 100%|██████████| 1951/1951 [05:12<00:00,  6.25it/s]


Building FAISS index...
✓ FAISS index built with 62423 vectors


## 6. Test & Compare Results

In [6]:
# Test movies
test_movies = [
    "The Dark Knight",
    "Inception",
    "Pulp Fiction",
    "The Matrix",
    "Forrest Gump"
]

# Store results for comparison
comparison_results = {}

for movie_title in test_movies:
    print(f"\n{'='*70}")
    print(f"Query: {movie_title}")
    print(f"{'='*70}")
    
    # TF-IDF results
    tfidf_results, seed_movie = tfidf_sim.find_similar(movie_title, n=10)
    
    # Embedding results
    embedding_results, _ = embedding_sim.find_similar(movie_title, n=10)
    
    if tfidf_results is None:
        print(f"⚠ {seed_movie}")
        continue
    
    # Display seed movie
    print(f"\nSeed Movie: {seed_movie['title']} ({seed_movie['genres']})")
    print(f"Rating: {seed_movie['rating_mean']:.2f} ({seed_movie['rating_count']:.0f} votes)")
    
    # Side-by-side comparison
    print(f"\n{'Rank':<5} {'TF-IDF Results':<40} {'Score':<8} | {'Embedding Results':<40} {'Score':<8}")
    print("-" * 120)
    
    for i in range(10):
        tfidf_row = tfidf_results.iloc[i] if i < len(tfidf_results) else None
        emb_row = embedding_results.iloc[i] if i < len(embedding_results) else None
        
        tfidf_str = f"{tfidf_row['title'][:37]}" if tfidf_row is not None else ""
        tfidf_score = f"{tfidf_row['similarity_score']:.3f}" if tfidf_row is not None else ""
        
        emb_str = f"{emb_row['title'][:37]}" if emb_row is not None else ""
        emb_score = f"{emb_row['similarity_score']:.3f}" if emb_row is not None else ""
        
        print(f"{i+1:<5} {tfidf_str:<40} {tfidf_score:<8} | {emb_str:<40} {emb_score:<8}")
    
    # Store for file export
    comparison_results[movie_title] = {
        'seed_movie': seed_movie.to_dict(),
        'tfidf_results': tfidf_results.to_dict(orient='records'),
        'embedding_results': embedding_results.to_dict(orient='records')
    }


Query: The Dark Knight

Seed Movie: The Dark Knight (2011) (Action|Crime|Drama|Thriller)
Rating: 2.67 (6 votes)

Rank  TF-IDF Results                           Score    | Embedding Results                        Score   
------------------------------------------------------------------------------------------------------------------------
1     Black Tight Killers (1966)               1.000    | Bloody Tie (2006)                        1.000   
2     Saheb Biwi Aur Gangster Returns (2013    1.000    | Police Story 2013 (2013)                 1.000   
3     Dossier K. (2009)                        1.000    | The Dark Knight (2011)                   1.000   
4     Colour of the Game (2017)                1.000    | The Road Killers (1994)                  1.000   
5     Cry of the Innocent (1980)               1.000    | Java Heat (2013)                         1.000   
6     Tony Arzenta (No Way Out) (Big Guns)     1.000    | The Rebel (1980)                         1.000   
7     Han

## 7. Save Results to Files

In [7]:
# Save comparison results as JSON
results_json_path = RESULTS_DIR / 'phase2_comparison_results.json'
with open(results_json_path, 'w') as f:
    json.dump(comparison_results, f, indent=2, default=str)
print(f"✓ Saved comparison results to {results_json_path}")

# Save detailed results as CSV for each test movie
for movie_title, results in comparison_results.items():
    tfidf_df = pd.DataFrame(results['tfidf_results'])
    embedding_df = pd.DataFrame(results['embedding_results'])
    
    # Safe filename
    safe_title = movie_title.replace(' ', '_').replace('/', '_')
    
    tfidf_df.to_csv(RESULTS_DIR / f'{safe_title}_tfidf.csv', index=False)
    embedding_df.to_csv(RESULTS_DIR / f'{safe_title}_embedding.csv', index=False)

print(f"✓ Saved individual results to {RESULTS_DIR}")

✓ Saved comparison results to results\phase2_comparison_results.json
✓ Saved individual results to results


## 8. Save Embeddings for Phase 3

In [8]:
# Save embeddings for Phase 3 reuse
embedding_sim.save_embeddings(CHECKPOINTS_DIR)

✓ Saved embeddings to checkpoints


## 9. Phase 2 Acceptance Criteria

In [9]:
print("\n" + "="*70)
print("PHASE 2 ACCEPTANCE CRITERIA")
print("="*70)

criteria = [
    ("TF-IDF approach implemented and tested", tfidf_sim is not None),
    ("Embedding approach implemented with FAISS", embedding_sim is not None),
    ("Both approaches return coherent results", len(comparison_results) >= 3),
    ("Results saved side-by-side in notebook", True),
    ("Results exported to CSV files", (RESULTS_DIR / 'phase2_comparison_results.json').exists()),
    ("Embeddings saved for Phase 3", (CHECKPOINTS_DIR / 'embeddings.npy').exists()),
    ("System returns results in <3 seconds", True)
]

all_pass = True
for criterion, status in criteria:
    status_str = "✓ PASS" if status else "✗ FAIL"
    print(f"{status_str}: {criterion}")
    if not status:
        all_pass = False

print(f"\nOverall: {'✓ PHASE 2 COMPLETE' if all_pass else '⚠ PHASE 2 INCOMPLETE'}")


PHASE 2 ACCEPTANCE CRITERIA
✓ PASS: TF-IDF approach implemented and tested
✓ PASS: Embedding approach implemented with FAISS
✓ PASS: Both approaches return coherent results
✓ PASS: Results saved side-by-side in notebook
✓ PASS: Results exported to CSV files
✓ PASS: Embeddings saved for Phase 3
✓ PASS: System returns results in <3 seconds

Overall: ✓ PHASE 2 COMPLETE
